### Load packages

In [89]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

import matplotlib
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.colors import to_rgba
from matplotlib.patches import Arc

import plotly.graph_objects as go
from graphviz import Digraph
from pyvis.network import Network
import webbrowser 

### Define parameters and file info

In [73]:
data_directory = r"..\data"

# -----------------------------------------------------------------------
# flow data
# -----------------------------------------------------------------------
flow_csv_file = r"MCN_nonprofit_economy_revenue_2023.csv"
flow_path = os.path.join(data_directory, flow_csv_file)
print("Nonprofit economy data CSV file:", flow_path)
if os.path.exists(flow_path):
    print("EXISTS")

# -----------------------------------------------------------------------
# Output files
# -----------------------------------------------------------------------

output_directory = r"..\output"

Nonprofit economy data CSV file: ..\data\MCN_nonprofit_economy_revenue_2023.csv
EXISTS


### Load in data

In [74]:
# Load nonprofit economy revenue data

flow_raw_df = pd.read_csv(flow_path)

# print(flow_raw_df.info())
flow_raw_df.head()

,Order,Source,Source Level,Recipient,Recipient Level,Amount
0,1,Donor-advised fund sponsors (national and comm...,2,"Arts, culture, humanities",3,3.42
1,2,Donor-advised fund sponsors (national and comm...,2,Education (minus colleges and universities),3,6.99
2,3,Donor-advised fund sponsors (national and comm...,2,Colleges and universities,3,6.82
3,4,Donor-advised fund sponsors (national and comm...,2,Environment and animals,3,3.41
4,5,Donor-advised fund sponsors (national and comm...,2,Health (minus hospitals and nursing homes),3,3.49


In [75]:
# Clean & format columns in nonprofit economy revenue data

flow_clean_df = flow_raw_df.copy()

flow_clean_df["Amount"] = pd.to_numeric(
    flow_clean_df["Amount"].replace("-", 0),
    errors="coerce"
)

flow_clean_df.head(13)

,Order,Source,Source Level,Recipient,Recipient Level,Amount
0,1,Donor-advised fund sponsors (national and comm...,2,"Arts, culture, humanities",3,3.42
1,2,Donor-advised fund sponsors (national and comm...,2,Education (minus colleges and universities),3,6.99
2,3,Donor-advised fund sponsors (national and comm...,2,Colleges and universities,3,6.82
3,4,Donor-advised fund sponsors (national and comm...,2,Environment and animals,3,3.41
4,5,Donor-advised fund sponsors (national and comm...,2,Health (minus hospitals and nursing homes),3,3.49
5,6,Donor-advised fund sponsors (national and comm...,2,Hospitals and nursing homes,3,2.15
6,7,Donor-advised fund sponsors (national and comm...,2,Human Services,3,9.38
7,8,Donor-advised fund sponsors (national and comm...,2,International/foreign affairs,3,3.14
8,9,Donor-advised fund sponsors (national and comm...,2,Public/societal benefit (minus national DAF sp...,3,4.98
9,10,Donor-advised fund sponsors (national and comm...,2,Foundations (minus community foundation DAF sp...,2,0.00


In [76]:
# Set color mapping
# Adapted from
# Colorblind-safe VincentvanGogh_1 palette from Regenbogen
# Source: https://github.com/tylerlittlefield/lisa

color_map_dict = {
    "Commercially Affiliated Sponsors": "#D95F30",
    "Community Foundations": "#DABD61",
    "Donation/Workplace Processors": "#6D8325",
    "Other Sponsors": "#8785B2"
}

In [77]:
%matplotlib inline
matplotlib.get_backend()

'inline'

### Build the graph with NetworkX

In [78]:
# Safety copy of dataframe
 
flow_df = flow_clean_df.copy()

# Build a directed weighted graph

G = nx.DiGraph()

for _, row in flow_df.iterrows():
    source = row["Source"]
    target = row["Recipient"]
    amount = row["Amount"]

    G.add_edge(source, target, weight=amount)

    G.nodes[source]["level"] = row["Source Level"]
    G.nodes[target]["level"] = row["Recipient Level"]

### Visualize with Graphviz (static charts)

In [79]:
dot = Digraph()
dot.attr(rankdir="LR")

for _, row in flow_df.iterrows():
    source = row["Source"]
    target = row["Recipient"]
    amount = row["Amount"]

    dot.edge(source, target, label=str(amount))

# Commented output out for now to reduce clutter
# dot.render("flow_graph", format="png")

### Visualize with Plotly Sankey (static charts)

In [86]:
sankey_df = flow_df.copy()

# clean
sankey_df["Amount"] = pd.to_numeric(sankey_df["Amount"], errors="coerce")
sankey_df = sankey_df.dropna(subset=["Amount"])

# merge duplicates
sankey_df = sankey_df.groupby(["Source", "Recipient"], as_index=False)["Amount"].sum()

# split loops
self_loops = sankey_df[sankey_df["Source"] == sankey_df["Recipient"]]
sankey_df = sankey_df[sankey_df["Source"] != sankey_df["Recipient"]]

# nodes + mapping
nodes = pd.Index(sankey_df["Source"].tolist() + sankey_df["Recipient"].tolist()).unique()
node_map = {n: i for i, n in enumerate(nodes)}

source_idx = sankey_df["Source"].map(node_map)
target_idx = sankey_df["Recipient"].map(node_map)
values = sankey_df["Amount"]

# sankey
fig = go.Figure(data=[go.Sankey(
    node=dict(
        label=list(nodes),
        pad=10,
        thickness=15
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values
    )
)])

# layout
fig.update_layout(
    title_text="Flow Diagram (Sankey)",
    font_size=10,
    height=900,
    width=900,
    margin=dict(l=20, r=20, t=40, b=20)
)

fig.show()

# self-loop summary (optional)
self_loop_summary = self_loops.groupby("Source")["Amount"].sum()

### Sankey with cosmetic fixes

In [98]:
# Safety copy of dataframe
poster_df = flow_df.copy()

# Ensure amounts are numbers
poster_df["Amount"] = pd.to_numeric(poster_df["Amount"], errors="coerce")
poster_df = poster_df.dropna(subset=["Amount"])

# Convert self-loops into retained nodes
loop_mask = poster_df["Source"] == poster_df["Recipient"]

poster_df.loc[loop_mask, "Recipient"] = (
    poster_df.loc[loop_mask, "Recipient"] + " (Retained)"
)

# Merge duplicates
poster_df = poster_df.groupby(["Source", "Recipient"], as_index=False)["Amount"].sum()

# Map the levels
level_map = (
    pd.concat([
        flow_df[["Source", "Source Level"]].rename(columns={"Source": "node", "Source Level": "level"}),
        flow_df[["Recipient", "Recipient Level"]].rename(columns={"Recipient": "node", "Recipient Level": "level"})
    ])
    .drop_duplicates()
    .groupby("node")["level"]
    .min()
)

# Add retained nodes
retained_nodes = [n for n in poster_df["Recipient"].unique() if "(Retained)" in n]

for node in retained_nodes:
    original = node.replace(" (Retained)", "")
    level_map[node] = level_map[original] + 0.3

# Build nodes and mappings
nodes = pd.Index(
    poster_df["Source"].tolist() +
    poster_df["Recipient"].tolist()
).unique()

node_map = {n: i for i, n in enumerate(nodes)}

source_idx = poster_df["Source"].map(node_map)
target_idx = poster_df["Recipient"].map(node_map)
values = poster_df["Amount"]

# Explicitly position the nodes
unique_levels = sorted(level_map.unique())
level_to_x = {
    lvl: i / (len(unique_levels)-1)
    for i, lvl in enumerate(unique_levels)
}

x_positions = [level_to_x[level_map[node]] for node in nodes]
y_positions = np.zeros(len(nodes))

# Spread vertically within each level
for lvl in unique_levels:
    level_nodes = [n for n in nodes if level_map[n] == lvl]
    ys = np.linspace(0.05, 0.95, len(level_nodes))

    for node, y in zip(level_nodes, ys):
        idx = node_map[node]
        y_positions[idx] = y

# Build Sankey plot with hidden labels
fig = go.Figure(data=[go.Sankey(
    arrangement="fixed",
    node=dict(
        label=[""] * len(nodes),   # hide labels
        x=x_positions,
        y=y_positions,
        pad=10,
        thickness=18
    ),
    link=dict(
        source=source_idx,
        target=target_idx,
        value=values
    )
)])

# Add labels as manual annotations
# Left side = right aligned
# Right side = left aligned
# Middle = centered
for node in nodes:
    idx = node_map[node]
    x = x_positions[idx]
    y = y_positions[idx]

    if x < 0.2:
        fig.add_annotation(
            x=x - 0.03,
            y=1-y,
            text=node,
            showarrow=False,
            xanchor="right",
            font=dict(size=11)
        )

    elif x > 0.8:
        fig.add_annotation(
            x=x + 0.03,
            y=1-y,
            text=node,
            showarrow=False,
            xanchor="left",
            font=dict(size=11)
        )

    else:
        fig.add_annotation(
            x=x,
            y=1-y,
            text=node,
            showarrow=False,
            xanchor="center",
            font=dict(size=10)
        )

# Style the layout
fig.update_layout(
    title_text="Flow Diagram",
    font_size=10,
    width=900,
    height=900,
    margin=dict(l=200, r=200, t=50, b=30)
)

# Show the plot
fig.show()

### Visualize with pyvis

In [91]:
net = Network(height="800px", width="100%", directed=True, notebook=False)

for node in nodes:
    net.add_node(node, label=node)

for _, row in sankey_df.iterrows():
    net.add_edge(
        row["Source"],
        row["Recipient"],
        value=row["Amount"],
        title=str(row["Amount"])
    )

for _, row in self_loops.iterrows():
    net.add_edge(
        row["Source"],
        row["Source"],
        value=row["Amount"],
        title=f"self-loop: {row['Amount']}"
    )

net.toggle_physics(False)

net.write_html("network.html")
webbrowser.open("network.html")

True